In [1]:
import sys
from pathlib import Path


PROJECT_ROOT = Path(r"C:\Users\Sherif\Desktop\ECG analysis")

sys.path.insert(0, str(PROJECT_ROOT))

In [14]:
import importlib
import src.train_pointnet
import src.pointcloud_ecg

importlib.reload(src.train_pointnet)
importlib.reload(src.pointcloud_ecg)

import torch
import matplotlib.pyplot as plt

from models.UNet1D import UNet1D
from models.pointnet import PointNetECG
from src.pointcloud_ecg import create_point_cloud
from src.dataset import create_dataloaders

In [3]:
train_loader, val_loader, test_loader, label_cols, _ = create_dataloaders()

In [18]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

unet = torch.load("unetecg.pth", map_location=device, weights_only=False)

unet.to(device)
unet.eval()

UNet1D(
  (down1): DownBlock(
    (conv): ConvBlock(
      (block): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (4): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
      )
    )
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (down2): DownBlock(
    (conv): ConvBlock(
      (block): Sequential(
        (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
        (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()


In [20]:
signals, _ = next(iter(train_loader))
signals = signals.to(device="cpu")

input_lead = signals[:, 0:1, :]

with torch.no_grad():
    _, embeddings = unet(input_lead)

print(embeddings.shape)

torch.Size([32, 12, 32])


In [21]:
point_cloud = create_point_cloud(signals, embeddings)

print(point_cloud.shape)  # (B, 12000, 3+d)

torch.Size([32, 12000, 35])


In [22]:
input_dim = point_cloud.shape[-1]

pointnet = PointNetECG(input_dim=input_dim).to(device)

out = pointnet(point_cloud)

print(out.shape)  # (B, 5)

torch.Size([32, 5])


In [15]:
from src.train_pointnet import train_pointnet

model = train_pointnet(model, train_loader, val_loader, epochs=10)

100%|██████████| 545/545 [22:26<00:00,  2.47s/it]


Epoch 1 | Train Loss: 0.4925
Validation Loss: 0.4303


100%|██████████| 545/545 [22:25<00:00,  2.47s/it]


Epoch 2 | Train Loss: 0.4210
Validation Loss: 0.4097


100%|██████████| 545/545 [34:40<00:00,  3.82s/it]


Epoch 3 | Train Loss: 0.4039
Validation Loss: 0.4025


... (10 outputs left out)

100%|██████████| 545/545 [20:03<00:00,  2.21s/it]


Epoch 9 | Train Loss: 0.3771
Validation Loss: 0.3846


100%|██████████| 545/545 [19:43<00:00,  2.17s/it]


Epoch 10 | Train Loss: 0.3744
Validation Loss: 0.3773


In [23]:
model.eval()

signals, labels = next(iter(test_loader))
signals = signals.to(device)
labels = labels.to(device)

input_lead = signals[:, 0:1, :]

with torch.no_grad():
    _, embeddings = unet(input_lead)
    pc = create_point_cloud(signals, embeddings)
    preds = model(pc)

print(preds.shape)

torch.Size([32, 5])


In [24]:
from src.train_pointnet import test_pointnet

# labels names 
print(label_cols)

# test
y_true, y_pred, macro_auc = test_pointnet(
    model=model,      # modèle entraîné
    unet=unet,           
    test_loader=test_loader,
    label_cols=label_cols,
    device="cpu"       
)


['CD', 'HYP', 'MI', 'NORM', 'STTC']


Testing PointNet: 100%|██████████| 69/69 [01:59<00:00,  1.73s/it]


===== POINTNET TEST RESULTS =====
Macro AUC: 0.8359

Classification Report:
              precision    recall  f1-score   support

          CD       0.74      0.49      0.59       496
         HYP       0.81      0.10      0.18       262
          MI       0.62      0.39      0.48       550
        NORM       0.75      0.88      0.81       963
        STTC       0.73      0.48      0.58       521

   micro avg       0.72      0.56      0.63      2792
   macro avg       0.73      0.47      0.52      2792
weighted avg       0.72      0.56      0.60      2792
 samples avg       0.61      0.58      0.58      2792


F1 (macro): 0.5243150791469746
Precision (macro): 0.7282466596727744
Recall (macro): 0.46518217678435814



c:\Users\Sherif\Desktop\ECG analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sherif\Desktop\ECG analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sherif\Desktop\ECG analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}